# Intent Gap — Stage A pilot (v2)

Expanded Stage A filter on a 50,000-conversation WildChat-1M sample. Differences from v1:
1. 45 repair phrases (vs.\ 17 in v1) including natural-language patterns observed in v1 hand review.
2. Scans every user turn after the first (vs.\ user turn 2 only in v1).
3. Tracks single-exchange conversations as a proxy for silent abandonment after intent-gap failure.

In [ ]:
!pip install -q datasets

## Repair-signal filter (v2 phrase list)

In [ ]:
import re

REPAIR_PHRASES = [
    # Explicit repair markers carried from v1
    r"no,?\s+i\s+(?:meant|said|asked)",
    r"that(?:'s|\s+is)\s+not\s+what\s+i\s+(?:asked|wanted|meant|said)",
    r"that(?:'s|\s+is)\s+not\s+(?:right|correct|it)",
    r"you\s+(?:misunderstood|missed|don'?t\s+understand|aren'?t\s+understanding)",
    r"you'?re\s+not\s+(?:getting|listening|understanding)",
    r"let\s+me\s+(?:rephrase|clarify|try\s+again)",
    r"to\s+be\s+clear,?\s+i\s+(?:want|meant|need)",
    r"actually,?\s+i\s+(?:want|meant|need|asked)",
    r"what\s+i\s+actually\s+(?:need|want|meant|asked)",
    r"the\s+question\s+was",
    r"my\s+question\s+(?:was|is)",
    r"you\s+didn'?t\s+answer",
    r"read\s+(?:the\s+)?(?:prompt|question|message)\s+again",
    r"i\s+(?:think|don'?t\s+think)\s+you\s+(?:misunderstood|understood|got\s+it)",
    r"that\s+wasn'?t\s+the\s+(?:question|point)",
    r"i\s+asked\s+(?:about|for|how|why|what|where|when)",
    r"this\s+is\s+(?:wrong|incorrect|not\s+what\s+i'?m\s+looking\s+for|not\s+helpful)",
    r"this\s+isn'?t\s+(?:right|what\s+i\s+(?:wanted|asked))",
    r"that(?:'s|\s+is)\s+(?:incorrect|wrong|not\s+accurate|inaccurate)",
    r"you\s+got\s+it\s+wrong",
    r"(?:not\s+useful|useless|unhelpful|not\s+helpful)",
    r"wrong\s+answer",
    # Natural-language patterns added in v2
    r"but\s+i\s+(?:wanted|asked|need|meant)",
    r"but\s+that(?:'s|\s+is)\s+not",
    r"i\s+meant",
    r"i\s+didn'?t\s+(?:ask|mean|want)",
    r"i\s+wasn'?t\s+asking",
    r"\bnot\s+quite\b",
    r"\bnope\b",
    r"^no[,.\s!?]*$",
    r"^wait[,.\s!?]",
    r"^ugh",
    r"^hmm",
    r"\bre[\s-]?read\b",
    r"try\s+again",
    r"do\s+(?:it|that)\s+(?:again|over)",
    r"that(?:'s|\s+is)\s+(?:still|now)\s+(?:wrong|not)",
    r"why\s+(?:are\s+you|did\s+you|do\s+you\s+keep)",
    r"stop\s+(?:doing|saying|trying)",
    r"that(?:'s|\s+is)\s+(?:still|just)\s+(?:not|wrong)",
    r"i\s+(?:already|just)\s+(?:said|told\s+you|asked)",
    r"can\s+you\s+(?:actually|just|please)",
    r"please\s+(?:just|actually)",
]
REPAIR_RE = re.compile('|'.join(f'({p})' for p in REPAIR_PHRASES),
                       flags=re.IGNORECASE | re.MULTILINE)


def find_repair(text: str):
    if not text:
        return None
    m = REPAIR_RE.search(text)
    return m.group(0).lower().strip() if m else None

## WildChat-1M stream

In [ ]:
from datasets import load_dataset

ds = load_dataset('allenai/WildChat-1M', split='train', streaming=True)

## Stage A v2 filter — 50,000-conversation sample

Scans every user turn after the first for any phrase from the v2 list. Also records single-exchange conversations (a user message followed by a single assistant response and no continuation) as an abandonment-rate proxy.

In [ ]:
import json
from collections import Counter

SAMPLE_SIZE = 50_000

n_total = 0
n_english = 0
n_long_enough = 0
n_repair_signal = 0
n_abandon_after_one = 0
phrase_counter = Counter()
candidates = []

for row in ds:
    if n_total >= SAMPLE_SIZE:
        break
    n_total += 1
    if row.get('language', '').lower() != 'english':
        continue
    n_english += 1
    turns = row.get('conversation', [])
    if len(turns) == 2:
        n_abandon_after_one += 1
        continue
    if len(turns) < 4:
        continue
    n_long_enough += 1
    repair_found = None
    repair_turn_idx = None
    matched_phrase = None
    for i, turn in enumerate(turns):
        if turn.get('role') == 'user' and i > 0:
            r = find_repair(turn.get('content', ''))
            if r:
                repair_found = turn.get('content', '')
                repair_turn_idx = i
                matched_phrase = r
                break
    if not repair_found:
        continue
    n_repair_signal += 1
    phrase_counter[matched_phrase] += 1
    prev_user = next((t['content'] for t in reversed(turns[:repair_turn_idx])
                      if t.get('role') == 'user'), '')
    prev_asst = next((t['content'] for t in reversed(turns[:repair_turn_idx])
                      if t.get('role') == 'assistant'), '')
    candidates.append({
        'conv_id': row.get('conversation_hash', ''),
        'repair_at_turn': repair_turn_idx,
        'matched_phrase': matched_phrase,
        'prev_user_prompt': prev_user[:1500],
        'prev_asst_response': prev_asst[:1500],
        'repair_turn': repair_found[:1500],
    })

print(f'Sample size:                  {n_total}')
print(f'English:                      {n_english}')
print(f'Single-exchange (abandoned):  {n_abandon_after_one}')
print(f'>= 4 turns:                   {n_long_enough}')
print(f'Repair signal hit:            {n_repair_signal}')
print(f'Repair rate (% of >=4-turn):  '
      f'{n_repair_signal / max(n_long_enough, 1) * 100:.3f}%')

## Phrase distribution

In [ ]:
for phrase, count in phrase_counter.most_common(25):
    print(f'{count:>4}   {phrase}')

## Serialize

In [ ]:
funnel = {
    'sample_size': n_total,
    'english': n_english,
    'single_exchange_abandoned': n_abandon_after_one,
    'at_least_4_turns': n_long_enough,
    'repair_signal_hit': n_repair_signal,
    'rate_per_4turn_english': round(n_repair_signal / max(n_long_enough, 1) * 100, 4),
    'top_phrases': phrase_counter.most_common(30),
    'source': 'allenai/WildChat-1M (streamed sample, v2 filter)',
    'stage': 'A v2 (expanded regex, all-turn scan)',
}

with open('pilot_funnel_v2.json', 'w') as f:
    json.dump(funnel, f, indent=2)

with open('pilot_examples_v2.jsonl', 'w') as f:
    for c in candidates[:500]:
        f.write(json.dumps(c) + '\n')